# ASL v1 — FULL-corpus pretrain (2731-class, ROI) + ROI fine-tune (Colab T4)

Pushes the proven data lever further: re-trains the encoder on **all 2731 glosses**
(~63.4k clips, ROI-cropped, val/test signers excluded) — vs the 1500 run that hit
73.3% test. Then fine-tunes the 75-class head. Resume-enabled for T4 disconnects.

### Upload **three files** to `MyDrive/asl-model/`
- `code_bundle.zip`            (asl package + configs + manifests/norms)
- `pretrain_2731_jpeg.npz`     (~4.5 GB - 2731-class ROI frames, JPEG-packed)
- `clips_roi.npz`              (~1 GB - 75-class ROI fine-tune frames)

Set **Runtime -> T4 GPU**, then run top to bottom.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os, zipfile, shutil
DRIVE = '/content/drive/MyDrive/asl-model'
os.makedirs('/content/work/artifacts/cache', exist_ok=True)
with zipfile.ZipFile(f'{DRIVE}/code_bundle.zip') as z:
    z.extractall('/content/work')
for f in ['pretrain_2731_jpeg.npz', 'clips_roi.npz']:
    shutil.copy(f'{DRIVE}/{f}', f'/content/work/artifacts/cache/{f}')
%cd /content/work
!pip -q install pyyaml onnx onnxruntime
import torch; print('cuda available:', torch.cuda.is_available())

In [ ]:
# Decode the 2731-class ROI JPEG cache back to frames.dat (~38 GB on Colab disk)
!PYTHONPATH=src python -m asl.pack_pretrain_jpeg --unpack \
    --in artifacts/cache/pretrain_2731_jpeg.npz --cache artifacts/cache/pretrain_2731

In [ ]:
# Pretrain encoder from scratch on the 2731-class ROI set. --resume continues from
# artifacts/checkpoints/pretrain/resume.pt if a session was cut off (harmless fresh).
!PYTHONPATH=src python -u -m asl.pretrain --cache artifacts/cache/pretrain_2731 \
    --norm artifacts/manifest/norm_roi.json \
    --epochs 45 --warmup 4 --batch-size 64 --lr 0.004 \
    --out artifacts/checkpoints/pretrain \
    --resume artifacts/checkpoints/pretrain/resume.pt
import shutil, os
os.makedirs('/content/drive/MyDrive/asl-model/out_2731', exist_ok=True)
shutil.copy('artifacts/checkpoints/pretrain/encoder.pt',
            '/content/drive/MyDrive/asl-model/out_2731/encoder.pt')
print('saved 2731-pretrain encoder.pt to Drive/out_2731')

In [ ]:
# Fine-tune the 75-class head on ROI clips + the 2731-pretrained encoder.
!PYTHONPATH=src python -u -m asl.train --config configs/finetune_roi.yaml
import shutil, os
OUT = '/content/drive/MyDrive/asl-model/out_2731'
for f in ['artifacts/checkpoints/finetune_roi/best.pt',
          'artifacts/checkpoints/finetune_roi/history.json']:
    shutil.copy(f, f'{OUT}/{os.path.basename(f)}')
print('2731 fine-tune done; best.pt + history.json copied to Drive/out_2731')